# 07 — Silver Layer

## Learning objectives

By the end, you can explain this topic, run a small RetailFlow example, inspect its result, and identify its production trade-offs.

- **Why it matters:** Use MERGE upserts for incremental keys in Delta-capable environments.
- **Prerequisites:** Python, SQL basics, and notebooks 00–06 where applicable.
- **Estimated time:** 60–90 minutes.
- **Compatibility:** ✅ Local PySpark baseline · 🟨 Delta requires compatible Delta runtime · 🟦 Databricks-only features are labeled.

## Conceptual explanation

Silver standardizes types, names, and business rules into trustworthy entity-level datasets.

Answer these questions as you read: what problem exists without this capability, what does Spark/Databricks do under the hood, and where can cost or correctness go wrong?

## Architecture / execution diagram

```mermaid
flowchart LR
  S[RetailFlow sources] --> B[Bronze: raw + metadata]
  B --> Q[Quality checks / quarantine]
  Q --> S1[Silver: clean entities]
  S1 --> G[Gold: business metrics]
  G --> C[SQL, dashboards, consumers]
```

Text fallback: sources → Bronze (preserve) → validate/clean → Silver (trusted rows) → Gold (metrics).

### What happens under the hood?
Spark builds a logical plan from your transformations. An action such as `show`, `count`, or `write` asks the optimizer to create physical stages and tasks. A shuffle moves data between executors and is often the expensive boundary.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while root != root.parent and not (root / "src").exists(): root = root.parent
sys.path.insert(0, str(root / "src"))
from retailflow.config import get_settings
settings = get_settings()
print(f"Project root: {root}")


## Runnable example

The following cell creates a local Spark session. On Databricks, `spark` already exists; use the existing session. `local[*]` is for learning only—it does not reproduce a distributed cluster.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.master("local[*]").appName("RetailFlowLearning").getOrCreate()

from retailflow.data_generation import generate_raw_data
generate_raw_data(root / "data" / "raw")
orders = spark.read.option("header", True).csv(str(root / "data" / "raw" / "orders.csv"))
print("Raw orders (includes one duplicate and one invalid timestamp):", orders.count())
orders.show(5, truncate=False)
assert orders.count() == 91


**Expected result:** the generated input contains 91 order rows. This includes intentional defects, so do not use that count as a clean business metric. If this assertion fails, delete `data/raw` and rerun the generator.

**Common mistake:** calling `collect()` to inspect a large DataFrame. It transfers all rows to the driver; use `show`, `limit`, aggregation, or a filtered sample instead.

In [ ]:
from pyspark.sql import functions as F
clean = (orders.withColumn("order_ts", F.to_timestamp("order_timestamp"))
    .filter(F.col("order_ts").isNotNull()).dropDuplicates(["order_id"]))
print("Valid, unique orders:", clean.count())
clean.groupBy("status").count().orderBy("status").show()
clean.explain()  # Look for the aggregation / exchange boundaries in the physical plan.


## Production note

In production, external storage and table governance are configured by your platform team. Do not depend on DBFS root or legacy mounts for new projects. With Unity Catalog, use approved external locations/volumes and three-level object names. Do not run destructive operations such as `VACUUM` without a retention and recovery policy.

## Silver implementation

Cast dates and numbers, standardize categories, deduplicate keys, validate relationships, and quarantine invalid rows before producing trusted entities.


## Try It Yourself

1. Change `dropDuplicates(["order_id"])` to `dropDuplicates()` and compare the count. Why do the results differ?
2. Call `clean.repartition(4).rdd.getNumPartitions()` and change 4 to 1 and 8. On small local data, timing is not meaningful; inspect the partition count instead.
3. Replace the filter with a quarantine DataFrame (`orders.filter(F.col("order_ts").isNull())`) and inspect bad rows.

## Exercises

### Beginner
1. Count orders for each status.
2. Find the invalid timestamp record.
3. Select only `order_id` and `customer_id`.

### Intermediate
4. Build a daily order-count table.
5. Join clean orders to order items and calculate completed revenue.

### Advanced challenge
6. Make the transformation idempotent and write a small quality-metrics table.

See `solutions/{prefix}_solutions.md` after attempting them.

## Knowledge check

### Conceptual
1. Which operation triggers Spark execution?
2. Why are schemas valuable at ingestion?
3. What is a shuffle?
4. Why retain Bronze data?
5. What does Delta add over ordinary files?

### Practical
1. Where would you inspect an execution plan?
2. How would you avoid duplicate key rows?
3. Which input should be quarantined here?

### Key takeaways
Spark plans lazily, quality is an explicit contract, and the Medallion layers have different responsibilities. Review transformations versus actions, schema casting, and joins before proceeding.